In [ ]:
!wget https://huggingface.co/datasets/barryallen16/java_vulnerability_dataset/resolve/main/java_vul_dataset.csv  # https://huggingface.co/datasets/barryallen16/java_vulnerability_dataset/blob/main/java_vul_dataset.csv # Grab the evaluation CSV (code + expected label) from HF (or your storage).

--2025-09-03 17:08:55--  https://huggingface.co/datasets/barryallen16/java_vulnerability_dataset/resolve/main/java_vul_dataset.csv
Resolving huggingface.co (huggingface.co)... 13.35.202.34, 13.35.202.121, 13.35.202.40, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.34|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/barryallen16/java_vulnerability_dataset/868c8f968215209f0f989c247099c5b7a6d9c777/java_vul_dataset.csv?%2Fdatasets%2Fbarryallen16%2Fjava_vulnerability_dataset%2Fresolve%2Fmain%2Fjava_vul_dataset.csv=&etag=%2264e5c6191056cdae6cba7fe96151f221431af074%22 [following]
--2025-09-03 17:08:56--  https://huggingface.co/api/resolve-cache/datasets/barryallen16/java_vulnerability_dataset/868c8f968215209f0f989c247099c5b7a6d9c777/java_vul_dataset.csv?%2Fdatasets%2Fbarryallen16%2Fjava_vulnerability_dataset%2Fresolve%2Fmain%2Fjava_vul_dataset.csv=&etag=%2264e5c6191056cdae6cba7fe96151f221431af074%22
Reusing e

In [ ]:
%%capture
# Hide noisy pip output to keep the notebook clean in the demo.
import os  # We install only what inference/eval needs. Unsloth is the library we use to load the Qwen2.5 model efficiently. # Used below to detect Colab vs local (optional)
if "COLAB_" not in "".join(os.environ.keys()):  # This checks if "COLAB_" exists in the environment variable keys, If "COLAB_" is not found, it means we not on Colab (so running locally)
    !pip install unsloth    # Colab already comes with most dependencies (CUDA, PyTorch, Transformers), so only unsloth is needed
else:                       # If we’re running locally or on another server, we need to install all required packages manually.
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
# "COLAB_" in os.environ.keys()
#bitsandbytes → enables 4-bit/8-bit quantization (saves VRAM).
#accelerate → helps with efficient training/inference.
#xformers → optimizes transformer layers.
#peft → Parameter-Efficient Fine-Tuning (LoRA).
#trl → Hugging Face’s training library (SFT).
#triton → GPU kernel optimizer.
#--no-deps → prevents pip from reinstalling dependencies (we control versions manually).sentencepiece → tokenizer for multilingual/code models. protobuf → serialization (needed by Transformers).

In [ ]:
import pandas as pd                    # Pandas is the standard tool for handling tabular data in Python (like CSV files).
df=pd.read_csv("java_vul_dataset.csv") # Each row = one test case: code-> java snippet,expected ->vuln/non vuln
df.head()                              # SHOWS first 5 rows of the DataFrame, Quick check that columns are loaded correctly and data looks structured.

,code,expected
0,import java.sql.*;\n public class Secur...,Classification: NOT_VULNERABLE. Vulnerability ...
1,import java.io.BufferedReader;\n import...,Classification: VULNERABLE. Vulnerability Type...
2,import java.util.Scanner;\n public clas...,Classification: VULNERABLE. Vulnerability Type...
3,import java.sql.*;\n public class Secur...,Classification: NOT_VULNERABLE. Vulnerability ...
4,To fix the Command Injection vulnerability in ...,Classification: NOT_VULNERABLE. Vulnerability ...


In [ ]:
df.columns  # Confirms dataset has exactly two columns: code and expected.

Index(['code', 'expected'], dtype='object')

In [ ]:
df.shape #Shows the dataset originally had 200 rows × 2 columns.

(200, 2)

In [ ]:
dataset_testcases=df[["code", "expected"]].to_dict(orient="records")

In [ ]:
dataset_testcases[0]

{'code': '    import java.sql.*;\n    public class SecureCode {\n    public static void main(String[] args) {\n    String userInput = "\'; DROP TABLE users; --"; // Malicious input\n    try {\n    Connection conn = DriverManager.getConnection("jdbc:mysql://localhost:3306/mydatabase", "username", "password");\n    // Using prepared statement to prevent SQL Injection\n    String query = "SELECT * FROM users WHERE username = ?";\n    PreparedStatement pstmt = conn.prepareStatement(query);\n    pstmt.setString(1, userInput);\n    ResultSet rs = pstmt.executeQuery();\n    while (rs.next()) {\n    System.out.println(rs.getString("username"));\n    }\n    conn.close();\n    } catch (SQLException e) {\n    e.printStackTrace();\n    }\n    }\n    }',
 'expected': 'Classification: NOT_VULNERABLE. Vulnerability Type: safe'}

In [ ]:
llm_df = pd.DataFrame({"code":[], "prediction":[]}) # This DataFrame stores LLM Outputs

In [ ]:
from unsloth import FastLanguageModel     # Loads FastLanguageModel from Unsloth which is high level helper class, give tools to load llm.
import torch
max_seq_length = 2048                     # Although Qwen2.5 supports longer contexts, 2048 is enough for our task, since most vulnerabilities are local to one method or a few connected methods.
dtype = None
load_in_4bit = True # LoRA (Low-Rank Adaptation) A specific method under PEFT. Works by inserting low-rank matrices (adapters) inside transformer layers.

model, tokenizer = FastLanguageModel.from_pretrained(       # Load the base Qwen2.5 7B Instruct model and its tokenizer.
    model_name = "barryallen16/Qwen2.5_7B_java_vul_final",  # As soon as call from_pretrained for the first time, the model files are copied to the local machine or Colab/VM storage.
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.8.0+cu126)
    Python  3.12.9 (you have 3.12.11)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.8.10: Fast Qwen2 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/80.8M [00:00<?, ?B/s]

Unsloth 2025.8.10 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
# model.save_pretrained("Qwen2.5_7B_java_vul_final")
# tokenizer.save_pretrained("Qwen2.5_7B_java_vul_final")

In [ ]:
# !zip -r Qwen2.5_7B_java_vul_final.zip Qwen2.5_7B_java_vul_final

In [ ]:
# @title
# "ask the model and get back the answer" function
def classify_code(code):  # Prepares a chat prompt for the LLM.
    messages = [
        {
            "role": "system",           # System: “You’re a classifier, follow strict format.” User: “Here’s the code.”
            "content": "You are a code vulnerability classifier. Analyze the provided Java code and classify it as either 'VULNERABLE' or 'NOT VULNERABLE'. If vulnerable, specify the exact vulnerability type from the dataset. Respond strictly in this format: 'Classification: <VULNERABLE/NOT VULNERABLE>. Vulnerability Type: <type>'. Do not include reasoning.",
        },
        {"role": "user", "content": code},
    ]

    inputs = tokenizer.apply_chat_template(   # Converts the structured chat into model input tokens
        messages,
        tokenize=True,
        add_generation_prompt=True, # tells the model it should generate a reply.
        return_tensors="pt",        # return as PyTorch tensors (needed for GPU).
        return_dict=True,
    ).to("cuda")                    # put those tensors on the GPU.

    outputs = model.generate(       # Uses the fine-tuned LLM to generate a response to the chat input
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=256
    )

    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)  # Decodes the generated token IDs (from the model) back into human-readable text.

    last_user_msg = messages[-1]["content"]  # the Java code we sent. Sometimes the decoded output includes the input code again.This line removes the code, keeping only the model’s answer.
    response = output_text.split(last_user_msg)[-1].strip() # cleans extra spaces.

    return response

In [ ]:

single_testcase=dataset_testcases[0]
model_prediction= classify_code(single_testcase['code'])
print(model_prediction)
print('expected: ',single_testcase['expected'])

assistant
Classification: NOT_VULNERABLE. Vulnerability Type: safe
expected:  Classification: NOT_VULNERABLE. Vulnerability Type: safe


In [ ]:
# @title
import json
from tqdm.notebook import tqdm
def evaluate_model(llm_name, test_cases):
    results = []           # empty list to store detailed info about every test case.
    correct = 0            # Counter for how many predictions were correct
    true_positives = 0     # cases where the code really was vulnerable and model said vulnerable.
    false_positives = 0    # safe code, but model raised false alarm (said vulnerable).
    false_negatives = 0    # vulnerable code, but model missed it (said safe).
    true_negatives = 0     # safe code, model correctly said safe.

    for i, test_case in enumerate(tqdm(test_cases, desc=f"Evaluating {llm_name}")): # tqdm = progress bar so you can see evaluation progress.
        code = test_case["code"]
        expected = test_case["expected"]  # The correct label ("VULNERABLE" or "NOT_VULNERABLE")
        try:
            prediction = classify_code(code) # Get the model's classification output as a string
            if "Classification: VULNERABLE" in prediction: # print("Model output:", prediction)
                predicted_class = "Classification: VULNERABLE"
            elif "Classification: NOT_VULNERABLE" in prediction:
                predicted_class = "Classification: NOT_VULNERABLE"
            else:
                predicted_class = "UNKNOWN"    # Handles any unexpected or badly formatted outputs

            new_row = pd.DataFrame([{"code": code, "prediction": predicted_class}]) # Creates a one-row DataFrame (new_row) with two columns:
            global llm_df               # Declares that you’re going to use the global variable llm_df inside this function.
            llm_df = pd.concat([llm_df, new_row], ignore_index=True) #Appends the new row (new_row) to the existing DataFrame (llm_df). ignore_index=True resets row numbering so it stays clean (0,1,2,…).
# Compare predicted class to expected, count as correct if their class matches
            if expected.startswith("Classification: VULNERABLE"):
                actual_class = "VULNERABLE"
            elif expected.startswith("Classification: NOT_VULNERABLE"):
                actual_class = "NOT_VULNERABLE"
            else:
                actual_class = "UNKNOWN"

            is_correct = (predicted_class == expected.split('.')[0].strip())  # Compare only the class part
            if is_correct:
                correct += 1 # Increment the count of correct predictions

# confusion matrix
            if actual_class == "VULNERABLE":
                if predicted_class == "Classification: VULNERABLE":
                    true_positives += 1     # Model found a vulnerability (TP)
                else:
                    false_negatives += 1    # Model missed a vulnerability (FN)
            elif actual_class == "NOT_VULNERABLE":
                if predicted_class == "Classification: VULNERABLE":
                    false_positives += 1    # Model raised a false alarm (FP)
                else:
                    true_negatives += 1     # Safe code in reality, model says not vulnerable.

            results.append({
                "test_case": i+1,
                "expected": expected,          # What should have been predicted
                "predicted": predicted_class,  # What the model actually predicted
                "correct": is_correct,
                "model_output": prediction     # The raw model output (may help with debugging)
            })

        except Exception as e:
            print(f"Error processing test case {i+1}: {str(e)}")
            results.append({
                "test_case": i+1,
                "error": str(e)
            })

    total_actual_vulnerable = true_positives + false_negatives     # All truly vulnerable cases
    total_actual_not_vulnerable = false_positives + true_negatives # All truly non-vulnerable cases

    accuracy = correct / len(test_cases) * 100 if len(test_cases) > 0 else 0  # Calculate accuracy as percentage of correct predictions

    # Calculate precision:TP / (TP + FP) fraction of predicted 'vulnerable' that were correct
    precision = true_positives / (true_positives + false_positives) * 100 if (true_positives + false_positives) > 0 else 0

    # Calculate recall: : TP / (TP + FN) fraction of all actual vulnerabilities detected
    recall = true_positives / (true_positives + false_negatives) * 100 if (true_positives + false_negatives) > 0 else 0

    # F1 Score: 2 * (Precision * Recall) / (Precision + Recall) # F1 Score is harmonic mean of precision and recall (percent)
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\n{llm_name} Testcases Evaluation Summary:")
    print(f"Total test cases: {len(test_cases)}")
    print(f"Correct predictions: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Precision: {precision:.2f}%")
    print(f"Recall: {recall:.2f}%")
    print(f"F1 Score: {f1_score:.2f}%")

    print("\nConfusion Matrix:")
    print(f"                Predicted Vulnerable  Predicted Not Vulnerable")
    print(f"Actual Vulnerable      {true_positives}                  {false_negatives}")
    print(f"Actual Not Vulnerable  {false_positives}                  {true_negatives}")

    output_file_path = llm_name + "_evaluation_results.json"
    with open(output_file_path, "w") as f:
        json.dump({
            "summary": {
                "total_test_cases": len(test_cases),
                "correct_predictions": correct,
                "accuracy": accuracy,     #Accuracy = (TP + TN) / total
                "precision": precision,   #Precision = TP / (TP + FP) → “When the model predicts vulnerable, how often is it right
                "recall": recall,         #Recall = TP / (TP + FN) → “Of all truly vulnerable cases, how many did the model catch?
                "f1_score": f1_score,     #F1-Score = Harmonic mean of precision and recall
                "class_distribution": {
                    "vulnerable": total_actual_vulnerable,
                    "not_vulnerable": total_actual_not_vulnerable
                },
                "confusion_matrix": {
                    "true_positives": true_positives,
                    "false_positives": false_positives,
                    "false_negatives": false_negatives,
                    "true_negatives": true_negatives
                }
            },
            "detailed_results": results
        }, f, indent=2)

    return results

True Positive (TP): Vulnerable code in reality, model also says vulnerable.

False Negative (FN): Vulnerable code in reality, model says not vulnerable.(missed)

False Positive (FP): Safe code in reality, model says vulnerable. (false alarm)

True Negative (TN): Safe code in reality, model says not vulnerable.

In [ ]:
results=evaluate_model("all testcases", dataset_testcases)

Evaluating all testcases:   0%|          | 0/200 [00:00<?, ?it/s]


all testcases Testcases Evaluation Summary:
Total test cases: 200
Correct predictions: 194
Accuracy: 97.00%
Precision: 96.81%
Recall: 96.81%
F1 Score: 96.81%

Confusion Matrix:
                Predicted Vulnerable  Predicted Not Vulnerable
Actual Vulnerable      91                  3
Actual Not Vulnerable  3                  103


In [ ]:
llm_df

,code,prediction
0,import java.sql.*;\n public class Secur...,Classification: NOT_VULNERABLE
1,import java.io.BufferedReader;\n import...,Classification: VULNERABLE
2,import java.util.Scanner;\n public clas...,Classification: VULNERABLE
3,import java.sql.*;\n public class Secur...,Classification: VULNERABLE
4,To fix the Command Injection vulnerability in ...,Classification: NOT_VULNERABLE
...,...,...
195,import org.owasp.encoder.Encode;\n publ...,Classification: NOT_VULNERABLE
196,import java.io.BufferedReader;\n import...,Classification: VULNERABLE
197,import java.sql.*;\n public class SQLIn...,Classification: NOT_VULNERABLE
198,import java.util.Scanner;\n public clas...,Classification: VULNERABLE


In [ ]:
list(dataset_testcases)[1]

{'code': '    import java.io.BufferedReader;\n    import java.io.IOException;\n    import java.io.InputStreamReader;\n    public class CommandInjectionVulnerability {\n    public static void main(String[] args) {\n    try {\n    BufferedReader reader = new BufferedReader(new InputStreamReader(System.in));\n    System.out.print("Enter a command to execute: ");\n    String command = reader.readLine();\n    Runtime runtime = Runtime.getRuntime();\n    Process process = runtime.exec("ping " + command);\n    BufferedReader inputReader = new BufferedReader(new InputStreamReader(process.getInputStream()));\n    String line;\n    while ((line = inputReader.readLine()) != null) {\n    System.out.println(line);\n    }\n    } catch (IOException e) {\n    e.printStackTrace();\n    }\n    }\n    }',
 'expected': 'Classification: VULNERABLE. Vulnerability Type: Command Injection'}